# Attention Pattern Statistics

Statistical characterization of attention patterns: entropy, concentration,
positional biases, stability, and diversity.

## Why This Matters

Attention pattern statistics reveals how heads route information between positions. Understanding attention mechanics is central to reverse-engineering transformer circuits, as attention patterns determine what information each component can access and transform.

**Key references:**
- [Elhage et al. (2021) "A Mathematical Framework for Transformer Circuits"](https://transformer-circuits.pub/2021/framework/index.html) — Foundational framework for attention head and residual stream analysis
- [Olsson et al. (2022) "In-context Learning and Induction Heads"](https://arxiv.org/abs/2209.11895) — Discovers induction heads and training phase transitions

In [ ]:
import jax
import jax.numpy as jnp
from irtk import HookedTransformer, HookedTransformerConfig
from irtk.attention_pattern_statistics import (
    attention_entropy_profile, attention_concentration_profile,
    attention_positional_bias, attention_pattern_stability,
    attention_head_diversity,
)

cfg = HookedTransformerConfig(
    n_layers=4, d_model=32, n_ctx=64, d_head=8, n_heads=4, d_vocab=100,
)
model = HookedTransformer(cfg)
key = jax.random.PRNGKey(42)
leaves, treedef = jax.tree.flatten(model)
new_leaves = []
for leaf in leaves:
    if isinstance(leaf, jnp.ndarray) and leaf.dtype == jnp.float32:
        key, subkey = jax.random.split(key)
        new_leaves.append(jax.random.normal(subkey, leaf.shape) * 0.1)
    else:
        new_leaves.append(leaf)
model = jax.tree.unflatten(treedef, new_leaves)
tokens = jnp.array([1, 15, 30, 45, 60, 75, 90])
print('Model ready')

## Entropy Profile

Is each head's attention focused or diffuse?

In [ ]:
result = attention_entropy_profile(model, tokens)
print(f"Focused heads: {result['n_focused']}/{len(result['per_head'])}\n")
for h in result['per_head']:
    focused = ' [FOCUSED]' if h['is_focused'] else ''
    bar = '█' * int(h['normalized_entropy'] * 20)
    print(f"  L{h['layer']}H{h['head']}: entropy={h['mean_entropy']:.4f}, "
          f"norm_ent={h['normalized_entropy']:.2f}{focused} {bar}")

## Concentration Profile

How much mass is concentrated in the top-1 and top-3 attended positions?

In [ ]:
result = attention_concentration_profile(model, tokens)
print(f"Sharp heads: {result['n_sharp']}/{len(result['per_head'])}\n")
for h in result['per_head']:
    sharp = ' [SHARP]' if h['is_sharp'] else ''
    print(f"  L{h['layer']}H{h['head']}: top1={h['mean_top1_mass']:.1%}, "
          f"top3={h['mean_top3_mass']:.1%}{sharp}")

## Positional Bias

Do heads prefer BOS, self, or previous-token attention?

In [ ]:
result = attention_positional_bias(model, tokens)
for h in result['per_head']:
    print(f"  L{h['layer']}H{h['head']}: bos={h['mean_bos_attention']:.3f}, "
          f"self={h['mean_self_attention']:.3f}, "
          f"prev={h['mean_prev_attention']:.3f} → {h['dominant_bias']}")

## Pattern Stability

Do attention patterns change smoothly or abruptly across positions?

In [ ]:
result = attention_pattern_stability(model, tokens)
print(f"Stable heads: {result['n_stable']}/{len(result['per_head'])}\n")
for h in result['per_head']:
    stable = ' [STABLE]' if h['is_stable'] else ''
    print(f"  L{h['layer']}H{h['head']}: diff={h['mean_consecutive_diff']:.4f}{stable}")

## Head Diversity

Are attention patterns diverse across heads within each layer?

In [ ]:
for layer in range(4):
    result = attention_head_diversity(model, tokens, layer=layer)
    diverse = 'diverse' if result['is_diverse'] else 'redundant'
    print(f"Layer {layer}: mean_sim={result['mean_abs_similarity']:.4f} [{diverse}]")
    for p in result['pairs'][:3]:
        print(f"  H{p['head_a']}↔H{p['head_b']}: {p['similarity']:+.4f}")
    print()